 # 配置与导入

In [1]:
import copy
import torch
import torch.nn as nn
import numpy as np
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.models.llama.modeling_llama import LlamaRotaryEmbedding
from kernel.palu_attention import apply_rotary_pos_emb
import lm_eval
from lm_eval.models.huggingface import HFLM
from lm_eval.tasks import TaskManager
from lm_eval.utils import make_table
# 超参
MODEL_PATH = "Meta-Llama-3-8B-Instruct_ratio-0.7_gs-4-fisher_uniform-whiten"
DATASET_NAME = "wikitext-2-raw-v1"  # wikitext-2-raw-v1
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
LR=5e-5


 # 评估函数

In [2]:
### PPL 评估
def evaluate_ppl(model, seqlen=2048, device="cuda", nsamples=None, input_ids=None):
    if input_ids is None: raise ValueError("evaluate_ppl now requires pre-tokenized input_ids to be passed in.")
    assert input_ids.dim() == 2, "input_ids must be 2D"

    if isinstance(device, str):
        device = torch.device(device)

    nsamples = input_ids.numel() // seqlen if nsamples is None else nsamples
    model.eval()

    nlls = []
    loss_fct = nn.CrossEntropyLoss()
    with torch.no_grad():
        for i in tqdm(range(nsamples)):
            batch = input_ids[:, (i * seqlen):((i + 1) * seqlen)].to(device)
            outputs = model(batch)
            logits = outputs.logits
            shift_logits = logits[:, :-1, :]
            shift_labels = input_ids[:, (i * seqlen):((i + 1) * seqlen)][:, 1:].to(device)
            loss = loss_fct(
                shift_logits.reshape(-1, shift_logits.size(-1)),
                shift_labels.reshape(-1)
            )
            neg_log_likelihood = loss.float() * seqlen
            nlls.append(neg_log_likelihood)

    ppl = torch.exp(torch.stack(nlls).sum() / (len(nlls) * seqlen)).item()
    return ppl

def example_generation(model, tokenizer, device):
    prompt = "Why research is so hard?"
    if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
        tokenizer.pad_token = tokenizer.eos_token

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    model.eval()
    with torch.no_grad():
        gen_ids = model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
            use_cache=False
        )

    gen_text = tokenizer.decode(gen_ids[0, inputs.input_ids.shape[1]:], skip_special_tokens=True)

    print("=== Example Prompt ===")
    print(prompt)
    print(gen_text)
    return

### Zero-shot OpenBookQA 准确率评估
def zero_shot_eval(model, tokenizer, tasks, *,
                                      batch_size: int = 8,
                                      max_length: int = 4096,
                                      limit: int | None = None,
                                      return_full: bool = False):
    """
    Same core logic as run_lm_eval.py but uses an already-loaded model/tokenizer.
    - Wraps model/tokenizer with HFLM
    - Runs lm_eval.simple_evaluate on the given tasks
    - Prints the results table and returns results['results'] by default
    - res = zero_shot_eval(model, tokenizer, tasks=["openbookqa"])
    """

    # normalize tasks
    task_list = [t.strip() for t in tasks.split(",")] if isinstance(tasks, str) else list(tasks)

    model.seqlen = max_length
    lm_obj = HFLM(pretrained=model, tokenizer=tokenizer, add_bos_token=False, batch_size=batch_size)
    task_manager = TaskManager()

    with torch.no_grad():
        results = lm_eval.simple_evaluate(
            model=lm_obj,
            tasks=task_list,
            task_manager=task_manager,
            log_samples=False,
            limit=limit,
        )

    print(make_table(results))
    return results if return_full else results["results"]


 ## 1) 加载model和dataset

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, torch_dtype=torch.float16, device_map="auto", use_cache=False
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("模型已加载。")

# 需要进行 HACK 的层（示例：修改为 [1, 2] 可对第 1、2 层依次处理）
hack_layer_ids = [2]
active_hack_layer_id = hack_layer_ids[0] if len(hack_layer_ids) > 0 else 0

# 存档 original layers
original_layers = {lid: copy.deepcopy(model.model.layers[lid].self_attn) for lid in hack_layer_ids}
print(f"original_layers 已存档: {list(original_layers.keys())}")

# 初始活动层 id（仅用于初始化变量；实际以循环内为准）

# 预缓存测试集以加速 PPL
ds_train = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
ds_test = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
test_texts = [ex["text"] for ex in ds_test if ex["text"].strip()]
test_text_cat = "\n\n".join(test_texts)
test_tok = tokenizer(test_text_cat, return_tensors="pt")
test_ids_all = test_tok.input_ids
test_ids_all = tokenizer("\n\n".join(ds_test["text"]), return_tensors="pt").input_ids
print("数据集已缓存。")
rotary_full = LlamaRotaryEmbedding(config=model.model.layers[0].self_attn.config).to(device=device, dtype=torch.float16)
# 重置指定层到 original
def reset_model(model, original_layers, layer_ids):
    layers = {}
    if isinstance(layer_ids, int):
        layer_ids = [layer_ids]
    for layer_id in layer_ids:
        model.model.layers[layer_id].self_attn = copy.deepcopy(original_layers[layer_id])
        layers[layer_id] = model.model.layers[layer_id].self_attn
        print(f"Reset layer {layer_id} to original")
    return layers


2025-09-18:12:09:31,691 INFO     [modeling.py:1005] We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

模型已加载。
original_layers 已存档: [2]
数据集已缓存。


 ## 2) 训练辅助函数



In [4]:
def sample_batch(tokenizer, batch_size=8, seq_len=128, device=device):
    texts = []
    while len(texts) < batch_size:
        t = ds_train[np.random.randint(len(ds_train))]["text"].strip()
        if t:
            texts.append(t)
    tok = tokenizer(
        texts, max_length=seq_len, truncation=True, padding="max_length", return_tensors="pt"
    )
    return tok.input_ids.to(device)

@torch.no_grad()
def capture_layer_input_hidden_states(model, input_ids, layer_id, device):
    """Capture the true hidden_states input to the target decoder layer via a forward pre-hook,
    and abort the rest of the forward immediately to save compute.
    """
    captured = {}
    class _StopForward(Exception): pass
    def _pre_hook(module, args):
        captured[layer_id] = args[0].detach()
        raise _StopForward()

    handle = model.model.layers[layer_id].register_forward_pre_hook(_pre_hook)
    assert model.config.use_cache is False, "use_cache must be False"
    model.eval()
    try:
        _ = model(input_ids.to(device))
    except _StopForward:
        pass
    finally:
        handle.remove()

    if layer_id not in captured: raise RuntimeError(f"Failed to capture hidden states at layer {layer_id}")

    return captured[layer_id]  # [B, T, H]

def alignment_loss(model, input_ids, layer_id, original_attn):
    B, T = input_ids.shape
    pos_ids = torch.arange(T, device=device).unsqueeze(0).expand(B, -1)

    # 获取真实输入到该层的 hidden states
    hs = capture_layer_input_hidden_states(model, input_ids, layer_id, device)  # [B, T, H]
    hs = model.model.layers[layer_id].input_layernorm(hs)
    # 计算 cos/sin（按 K 的 head_dim）
    hack_attn = model.model.layers[layer_id].self_attn
    head_dim = hack_attn.head_dim
    num_kv = hack_attn.num_key_value_heads
    dummy = torch.empty(B, num_kv, T, head_dim, device=device, dtype=hs.dtype)
    cos, sin = rotary_full(dummy, pos_ids)  # 形状 [B, T, head_dim]

    # === 目标：PALU 路径（RoPE(x@U@V)) ===
    with torch.no_grad():
        k_lat_palu = original_attn.k_proj.project_to_latent(hs)  # [B, T, total_latent_k]
        k_palu = original_attn.k_proj.reconstruct(k_lat_palu)    # [B, T, num_kv*head_dim]
        k_palu = k_palu.view(B, T, num_kv, head_dim).transpose(1, 2)  # [B, heads, T, head_dim]
        _, k_palu_rope = apply_rotary_pos_emb(None, k_palu, cos, sin)  # 对 key 施加 RoPE

    # === 预测：HACK 路径（RoPE(x@U)@V) ===
    k_lat_hack = hack_attn.k_proj.project_to_latent(hs.float())  # [B, T, total_latent_k]
    latent_dim = k_lat_hack.shape[-1] // num_kv
    k_lat_hack = k_lat_hack.view(B, T, num_kv, latent_dim).transpose(1, 2)  # [B, heads, T, latent_dim]
    # 在 latent 维度上截断 cos/sin 后应用 RoPE
    _, k_lat_hack_rope = apply_rotary_pos_emb(None, k_lat_hack, cos[..., :latent_dim], sin[..., :latent_dim])
    # 重构回 key states
    k_lat_hack_rope = k_lat_hack_rope.transpose(1, 2).reshape(B, T, -1)  # [B, T, total_latent_k]
    k_hack = hack_attn.k_proj.reconstruct(k_lat_hack_rope).view(B, T, num_kv, head_dim).transpose(1, 2)  # [B, heads, T, head_dim]

    # MSE 对齐
    return nn.functional.mse_loss(k_hack, k_palu_rope.float())


 ## 3) 训练循环：优化 U/V（仅 K），并周期性评估整模 PPL

In [ ]:
SEQ_LEN = 2048
BATCH_SIZE = 8
NUM_STEPS = 2000
EVAL_EVERY = 200
MAX_TEST_WINDOWS = 10  # 每次快速 PPL 评估用多少个窗口
# fix random seed
torch.manual_seed(42)
np.random.seed(42)
best_layers = {}
reset_model(model, original_layers, layer_ids=hack_layer_ids)
# 逐层进行 HACK/finetune/验证
for active_hack_layer_id in hack_layer_ids:
    print(f"\n===== Processing layer {active_hack_layer_id} =====")

    # 初始 PPL
    base_ppl = evaluate_ppl(model, SEQ_LEN, device=device, input_ids=test_ids_all, nsamples=10)
    print(f"Baseline before finetune (layer={active_hack_layer_id}), PPL: {base_ppl:.4f}")

    # 初始 HACK Attention PPL
    hack_attn = model.model.layers[active_hack_layer_id].self_attn
    original_attn = copy.deepcopy(original_layers[active_hack_layer_id])
    hack_attn.rope_latent = True  # Set HACK Attention RoPE(x@U)@V
    base_ppl_hack = evaluate_ppl(model, SEQ_LEN, device=device, input_ids=test_ids_all, nsamples=10)
    print(f"HACK Attention (layer={active_hack_layer_id}, rope = {hack_attn.rope_latent}) PPL: {base_ppl_hack:.4f}")

    # 训练
    loss_hist = []
    ppl_hist = []
    best_ppl = float('inf')
    best_attn = None
    hack_attn.to(dtype=torch.float32)
    train_params = [hack_attn.k_proj.VT.weight, hack_attn.k_proj.U[0].weight, hack_attn.k_proj.U[1].weight]
    for n, p in hack_attn.named_parameters(): p.requires_grad_(False)
    for p in train_params: p.requires_grad_(True)  # 只训练当前层的 k_proj 中的 U 和 VT
    init_params = [p.detach().clone() for p in train_params]
    optimizer = torch.optim.AdamW(train_params, lr=LR, weight_decay=1e-6, eps=1e-8)
    print("构建训练目标 train_params")
    for step in tqdm(range(1, NUM_STEPS + 1), desc=f"Aligning K at layer {active_hack_layer_id} (RoPE latent vs full)"):
        input_ids = sample_batch(tokenizer, BATCH_SIZE, SEQ_LEN, device)
        optimizer.zero_grad(set_to_none=True)
        # loss = alignment_loss(model, input_ids, active_hack_layer_id, original_attn)
        lambda_reg = 1e-5
        align = alignment_loss(model, input_ids, active_hack_layer_id, original_attn)
        reg = sum(torch.sum((p - p_init).pow(2)) for p, p_init in zip(train_params, init_params))
        loss = align + lambda_reg * reg
        if torch.isfinite(loss):
            loss.backward()
            torch.nn.utils.clip_grad_norm_(train_params, 0.05)
            optimizer.step()
            loss_hist.append(float(loss.item()))
        if step % EVAL_EVERY == 0:
            hack_attn_eval = copy.deepcopy(hack_attn).to(torch.float16)
            hack_attn_eval.eval()
            model.model.layers[active_hack_layer_id].self_attn = hack_attn_eval
            ppl =  evaluate_ppl(model, SEQ_LEN, device=device, nsamples=10, input_ids=test_ids_all)
            model.model.layers[active_hack_layer_id].self_attn = hack_attn
            ppl_hist.append(ppl)
            print(f"Step {step}: align_loss={loss.item():.6e}, evaluate_ppl with 10 samples={ppl:.4f}")
            if ppl < best_ppl:
                best_ppl = ppl
                best_attn = copy.deepcopy(model.model.layers[active_hack_layer_id].self_attn)

    # 结束后做一次完整 PPL（注入最优权重）
    if best_attn is not None:
        model.model.layers[active_hack_layer_id].self_attn = best_attn.to(torch.float16)
    final_ppl = evaluate_ppl(model, SEQ_LEN, device=device, input_ids=test_ids_all)
    print(f"Final (HACK@layer{active_hack_layer_id}) PPL: {final_ppl:.4f}")
    example_generation(model, tokenizer, device)
    print("\n--------------------------------------------------------------------------------------------------------------------------------\n")

    # 记录最佳层权重
    best_layers[active_hack_layer_id] = copy.deepcopy(model.model.layers[active_hack_layer_id].self_attn)


Reset layer 2 to original

===== Processing layer 2 =====


100%|████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:01<00:00,  5.06it/s]


Baseline before finetune (layer=2), PPL: 8.4081


100%|████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:01<00:00,  5.06it/s]


HACK Attention (layer=2, rope = True) PPL: 61.8920
构建训练目标 train_params


Aligning K at layer 2 (RoPE latent vs full):  10%|███▌                                | 201/2000 [00:22<15:37,  1.92it/s]

Step 200: align_loss=5.551579e+00, evaluate_ppl with 10 samples=244.3099


Aligning K at layer 2 (RoPE latent vs full):  20%|███████▏                            | 399/2000 [00:42<02:43,  9.79it/s]

 ## 评估 PPL & OpenBookQA



In [ ]:
###===============ppl评估 & zero-shot OpenBookQA（逐层）===============###
# Original
attn_layers = reset_model(model, original_layers, hack_layer_ids)
ppl_original = evaluate_ppl(model, SEQ_LEN, device=device, input_ids=test_ids_all)
res_original = zero_shot_eval(model, tokenizer, tasks=["openbookqa"])
print(f"👉🏻👉🏻👉🏻👉🏻Evaluating Original LLAMA: PPL= {ppl_original:.4f}")
example_generation(model, tokenizer, device)
print("\n--------------------------------------------------------------------------------------------------------------------------------\n")

# HACK before finetune
for layer_id in hack_layer_ids:
    attn_layers[layer_id].rope_latent = True
ppl_hack = evaluate_ppl(model, SEQ_LEN, device=device, input_ids=test_ids_all)
res_hack = zero_shot_eval(model, tokenizer, tasks=["openbookqa"])
print(f"👉🏻👉🏻👉🏻👉🏻Evaluating LLAMA with HACK on layer {hack_layer_ids} before finetune: PPL= {ppl_hack:.4f}")
example_generation(model, tokenizer, device)
print("\n--------------------------------------------------------------------------------------------------------------------------------\n")

# HACK after finetune (inject best)
for layer_id in hack_layer_ids:
    model.model.layers[layer_id].self_attn = best_layers[layer_id]
final_ppl = evaluate_ppl(model, SEQ_LEN, device=device, input_ids=test_ids_all)
res_hack_finetune = zero_shot_eval(model, tokenizer, tasks=["openbookqa"])
print(f"👉🏻👉🏻👉🏻👉🏻Evaluating LLAMA with HACK on layer {hack_layer_ids} after finetune: PPL= {final_ppl:.4f}")
example_generation(model, tokenizer, device)
print("\n--------------------------------------------------------------------------------------------------------------------------------\n")


 ### Save Best Layer

In [ ]:
for layer_id, best_attn in best_layers.items():
    torch.save(best_attn.state_dict(), f"HACKDump/{MODEL_PATH}_layer{layer_id}.pt")